# CMV Human Annotation Notebook

This notebook collects pairwise human judgments for three annotation tasks: **Cogency**, **Expected Rhetorical Effectiveness**, and **Evidential Support**.

The workflow is split into separate sections:

1. **Initial setup** — choose your annotator ID and load the shared sample.
2. **Instructions and worked examples** — read these before annotating.
3. **Calibration round** — everyone independently annotates the same first 5 OPs (75 response-pair assignments), then submits the calibration CSV.
4. **Calibration revision (Optional)** — use this only if the research team asks you to revisit calibration answers after reviewing agreement and clarifying the instructions.
5. **Main annotation round** — annotate the remaining 115 OPs (1,725 response-pair assignments).
6. **What to submit** — the exact files to return when annotation is complete.
7. **Progress and output files** — check completion and saved-file locations.

Expected folder structure:

Inside this repository the notebook reads `data/annotation_input.csv` and saves to `annotation/output/` (ignored by git). The notebook saves continuously. Do not rename or manually edit files in the output folder while annotation is in progress.

**About the released labels.** The A/B order of each pair is derived from a hash that includes the annotator ID, so re-running this notebook with a new ID shows different A/B orders. The released label files in `data/human/` record the order each annotator actually saw (`response_id_a`, `response_id_b`, `ab_flipped`); the analysis restates every label with A = the lower response ID before comparing anything. Main-round labels were collected with a standalone HTML version of this same interface (same instructions, questions, labels and A/B assignment), which is not included in the repository.


# 1. Initial setup

Set `ANNOTATOR_ID` below before running anything else. Each annotator must use a unique ID with no spaces.

Run the notebook from inside the cloned repository (locally in Jupyter, or in Colab after `!git clone <repo-url>` and `%cd <repo-folder>`). Install `ipywidgets` if it is not available.

In [ ]:
# ============================================================
# SETUP AND DATA LOADING
# ============================================================

ANNOTATOR_ID = "H1"   # <<< CHANGE THIS. A unique ID with no spaces, e.g. H1, H2, H3
PROTOCOL_VERSION = "cmv_annotation_v4_2026_09_24"

import os, sys, json, html, hashlib, itertools, time, re
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown, clear_output

assert ANNOTATOR_ID and " " not in ANNOTATOR_ID, "Set a non-empty ANNOTATOR_ID with no spaces."

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    try:
        from google.colab import output as colab_output
        colab_output.enable_custom_widget_manager()
    except Exception:
        pass

# Repository root: the first folder at or above the working directory that contains data/annotation_input.csv.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "annotation_input.csv").exists())
INPUT_DIR = ROOT / "data"
ANNO_DIR  = ROOT / "annotation" / "output"
ANNO_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_CSV = INPUT_DIR / "annotation_input.csv"

# utf-8-sig reads both ordinary UTF-8 and UTF-8 files with a BOM.
raw = pd.read_csv(SAMPLE_CSV, dtype={"post_id": str, "response_id": str}, encoding="utf-8-sig")

SAFE_COLS = ["post_id", "response_id", "op_title", "op_text_clean", "response_text_clean"]
missing = [c for c in SAFE_COLS if c not in raw.columns]
assert not missing, f"Missing expected columns: {missing}"

# Preserve the fixed sample order if supplied; otherwise use first appearance in the file.
if "sample_post_order" in raw.columns:
    post_order_map = (raw[["post_id", "sample_post_order"]]
                      .drop_duplicates("post_id")
                      .set_index("post_id")["sample_post_order"].astype(int).to_dict())
else:
    post_order_map = {pid: i for i, pid in enumerate(raw["post_id"].drop_duplicates().tolist())}

# Whitelist only annotation-safe columns. Behavioral outcomes are not retained in memory.
data = raw[SAFE_COLS].copy()
del raw
for c in ("op_title", "op_text_clean", "response_text_clean"):
    data[c] = data[c].fillna("").astype(str)
data["sample_post_order"] = data["post_id"].map(post_order_map).astype(int)

assert len(data) == 720, f"Expected 720 responses; found {len(data)}"
assert data["post_id"].nunique() == 120, f"Expected 120 OPs; found {data['post_id'].nunique()}"
assert (data.groupby("post_id").size() == 6).all(), "Every OP must have exactly 6 responses."
assert not data.duplicated(["post_id", "response_id"]).any(), "Duplicate post/response rows found."

# Encoding diagnostic. These are common signs that UTF-8 was decoded incorrectly.
mojibake_tokens = ("â€™", "â€œ", "â€", "Ã", "Â", "")
joined_text = "\n".join(data["op_text_clean"].tolist() + data["response_text_clean"].tolist())
hits = {tok: joined_text.count(tok) for tok in mojibake_tokens if tok in joined_text}
if hits:
    print("WARNING: possible mojibake detected:", hits)
else:
    print("Encoding check: no common mojibake signatures detected.")

OUT_CSV          = ANNO_DIR / f"annotations_{ANNOTATOR_ID}.csv"
HISTORY_CSV      = ANNO_DIR / f"annotation_history_{ANNOTATOR_ID}.csv"
CAL_BASELINE_CSV = ANNO_DIR / f"calibration_baseline_{ANNOTATOR_ID}.csv"
CAL_CURRENT_CSV  = ANNO_DIR / f"calibration_current_{ANNOTATOR_ID}.csv"
PROTOCOL_JSON    = ANNO_DIR / f"annotation_protocol_{PROTOCOL_VERSION}.json"
MIRROR_CSV       = (Path("/content") / f"annotations_{ANNOTATOR_ID}_mirror.csv") if IN_COLAB else None

print(f"Project root : {ROOT}")
print(f"Input file   : {SAMPLE_CSV}")
print(f"Annotator    : {ANNOTATOR_ID}")
print(f"Output dir   : {ANNO_DIR}")

In [ ]:
# ============================================================
# FIXED TASK SETS + STORAGE
# ============================================================

def _h(*parts) -> int:
    """Stable cross-session hash. Do not use Python's salted built-in hash()."""
    s = "||".join(str(p) for p in parts)
    return int(hashlib.sha256(s.encode("utf-8")).hexdigest()[:16], 16)

MASTER_POST_IDS = [
    pid for pid, _ in sorted(post_order_map.items(), key=lambda kv: (kv[1], kv[0]))
]
assert len(MASTER_POST_IDS) == 120

# Deterministic calibration set: the same first five sampled OPs for every annotator.
CAL_POST_IDS  = MASTER_POST_IDS[:5]
MAIN_POST_IDS = MASTER_POST_IDS[5:]

by_post = {
    pid: g.sort_values("response_id").reset_index(drop=True)
    for pid, g in data.groupby("post_id")
}
TEXT = {r.response_id: r.response_text_clean for r in data.itertuples()}
OP   = {pid: (g.op_title.iloc[0], g.op_text_clean.iloc[0]) for pid, g in data.groupby("post_id")}


def build_tasks(post_ids, phase, shuffle_posts):
    ids = list(post_ids)
    if shuffle_posts:
        ids = sorted(ids, key=lambda p: _h("postorder", ANNOTATOR_ID, p))

    tasks = []
    for phase_post_order, pid in enumerate(ids):
        g = by_post[pid]
        rids = list(g.response_id)
        pairs = list(itertools.combinations(rids, 2))  # 15 unordered pairs
        pairs = sorted(
            pairs,
            key=lambda pr: _h("pairorder", ANNOTATOR_ID, phase, pid, pr[0], pr[1])
        )
        for pair_in_post, (lo, hi) in enumerate(pairs):
            flip = _h("orient", ANNOTATOR_ID, pid, lo, hi) % 2 == 1
            rid_a, rid_b = (hi, lo) if flip else (lo, hi)
            tasks.append({
                "pair_key": f"{pid}|{lo}|{hi}",
                "phase": phase,
                "post_id": pid,
                "sample_post_order": int(post_order_map[pid]),
                "phase_post_order": int(phase_post_order),
                "pair_in_post": int(pair_in_post),
                "response_id_a": rid_a,
                "response_id_b": rid_b,
                "ab_flipped": bool(flip),
                "task_order": "cogency_first",
            })
    return tasks

CAL_TASKS  = build_tasks(CAL_POST_IDS,  "calibration", shuffle_posts=False)
MAIN_TASKS = build_tasks(MAIN_POST_IDS, "main",        shuffle_posts=True)
ALL_TASKS  = CAL_TASKS + MAIN_TASKS

assert len(CAL_TASKS) == 75
assert len(MAIN_TASKS) == 1725
assert len(ALL_TASKS) == 1800
assert {t["post_id"] for t in CAL_TASKS} == set(CAL_POST_IDS)

print("Fixed calibration OP IDs:")
for i, pid in enumerate(CAL_POST_IDS, 1):
    print(f"  {i}. {pid}")
print(f"\nCalibration: {len(CAL_TASKS)} pairs across 5 OPs")
print(f"Main round : {len(MAIN_TASKS)} pairs across 115 OPs")

RECORD_COLS = [
    "pair_key", "annotator_id", "instruction_version", "phase_of_last_edit",
    "post_id", "sample_post_order", "phase_post_order", "pair_in_post",
    "response_id_a", "response_id_b", "ab_flipped", "task_order",
    "cogency_choice", "cogency_winner_response_id", "cogency_margin", "cogency_signed",
    "effect_choice", "effect_winner_response_id", "effect_margin", "effect_signed",
    "evidence_a_present", "evidence_b_present", "evidence_presence_pattern",
    "evidence_choice", "evidence_winner_response_id", "evidence_margin", "evidence_signed",
    "a_nonargumentative", "b_nonargumentative", "a_unreadable", "b_unreadable",
    "notes", "seconds_on_pair_total", "revision_count",
    "first_submitted_utc", "last_modified_utc",
]

HISTORY_COLS = [
    "event_index", "event_type", "event_phase", "event_utc",
] + RECORD_COLS


def _atomic_write(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(path) + ".tmp")
    try:
        df.to_csv(tmp, index=False, encoding="utf-8-sig")
        os.replace(tmp, path)
    except OSError:
        df.to_csv(path, index=False, encoding="utf-8-sig")
        if tmp.exists():
            try:
                tmp.unlink()
            except Exception:
                pass


def _load_dict_csv(path):
    if not path.exists():
        return {}
    try:
        d = pd.read_csv(path, dtype={"post_id": str, "pair_key": str, "response_id_a": str,
                                    "response_id_b": str}, encoding="utf-8-sig")
        return {r["pair_key"]: r for r in d.to_dict("records")}
    except Exception as e:
        print(f"Could not read {path.name}: {e}")
        return {}


def _load_history(path):
    if not path.exists():
        return []
    try:
        return pd.read_csv(path, dtype={"post_id": str, "pair_key": str},
                           encoding="utf-8-sig").to_dict("records")
    except Exception as e:
        print(f"Could not read {path.name}: {e}")
        return []

RECORDS = _load_dict_csv(OUT_CSV)
if not RECORDS and MIRROR_CSV is not None:
    RECORDS = _load_dict_csv(MIRROR_CSV)
HISTORY = _load_history(HISTORY_CSV)


def current_calibration_df():
    keys = [t["pair_key"] for t in CAL_TASKS]
    rows = [RECORDS[k] for k in keys if k in RECORDS]
    if not rows:
        return pd.DataFrame(columns=RECORD_COLS)
    d = pd.DataFrame(rows)
    for c in RECORD_COLS:
        if c not in d.columns:
            d[c] = pd.NA
    return d[RECORD_COLS].sort_values(["sample_post_order", "pair_in_post"])


def save_records():
    rows = list(RECORDS.values())
    d = pd.DataFrame(rows)
    if len(d):
        for c in RECORD_COLS:
            if c not in d.columns:
                d[c] = pd.NA
        d = d[RECORD_COLS].sort_values(["sample_post_order", "pair_in_post"])
    else:
        d = pd.DataFrame(columns=RECORD_COLS)
    _atomic_write(d, OUT_CSV)
    if MIRROR_CSV is not None:
        _atomic_write(d, MIRROR_CSV)

    h = pd.DataFrame(HISTORY)
    if len(h):
        for c in HISTORY_COLS:
            if c not in h.columns:
                h[c] = pd.NA
        h = h[HISTORY_COLS]
    else:
        h = pd.DataFrame(columns=HISTORY_COLS)
    _atomic_write(h, HISTORY_CSV)

    # Once a calibration baseline exists, maintain a separate current calibration snapshot.
    if CAL_BASELINE_CSV.exists():
        _atomic_write(current_calibration_df(), CAL_CURRENT_CSV)


def completed(tasks):
    return sum(t["pair_key"] in RECORDS for t in tasks)

print(f"Resumed records: {len(RECORDS)} / 1800")
print(f"Calibration    : {completed(CAL_TASKS)} / 75")
print(f"Main round     : {completed(MAIN_TASKS)} / 1725")


# 2. Instructions and worked examples

Read this section completely before beginning the calibration round. The same definitions and decision rules apply throughout the notebook.

The worked examples below are **synthetic examples created only to explain the annotation tasks**. They are not drawn from the 120 OP annotation sample.


In [ ]:
# ============================================================
# ANNOTATION PROTOCOL
# ============================================================

PROTOCOL = {
    "version": PROTOCOL_VERSION,
    "overview": (
        "In each annotation, you will see one original r/ChangeMyView post and two direct responses "
        "to that post, labeled Response A and Response B. You will make three judgments in this order: "
        "Cogency, Expected Rhetorical Effectiveness, and Evidential Support. Judge only the material "
        "shown to you. Do not search the web, use AI assistants, look up the users, or use information "
        "from later discussion. Base your judgments about the OP on the original post and reasonable inferences "
        "from what the OP says. Do not assume facts about the OP that are unsupported by the post or use "
        "outside information about the user. Judge each displayed pair on its own. Do not revisit an earlier "
        "answer merely to make your answers form a perfectly consistent ranking across all responses."
    ),
    "scale_intro": (
        "For Cogency and Expected Rhetorical Effectiveness, choose exactly one of four positions on the "
        "comparative scale: Strong A, Weak A, Weak B, or Strong B. Every position selects either A or B; "
        "the Strong/Weak distinction records how large the difference seems. Use Strong when one response "
        "is clearly better on the question being asked. Use Weak when one response is only somewhat better "
        "or you have only a slight preference. There is no tie option. If the responses seem nearly equal, "
        "choose the side you lean toward and use Weak. Strong/Weak describes the strength of the comparative "
        "preference, not your confidence in your own judgment. The same four-way scale appears for Evidential "
        "Support only when both responses are marked as providing evidence."
    ),
    "scale": {
        "Strong A": "Response A is clearly better on the target question.",
        "Weak A": "Response A is somewhat better, or you only slightly prefer A.",
        "Weak B": "Response B is somewhat better, or you only slightly prefer B.",
        "Strong B": "Response B is clearly better on the target question."
    },
    "cogency": {
        "title": "COGENCY — QUALITY OF THE ARGUMENT",
        "full": (
            "A cogent argument offers reasons that adequately support its conclusion. Judge the argument "
            "advanced by the response itself. Consider three aspects: Acceptability — are the premises "
            "reasonable enough to serve as support for the conclusion? Relevance — do the premises actually "
            "bear on the conclusion? Sufficiency — taken together, do the premises provide enough support for "
            "drawing the conclusion? Do not judge cogency by whether you personally agree with the conclusion "
            "or by whether this particular OP is likely to be persuaded. Use the material shown and ordinary "
            "background knowledge. Do not conduct external fact-checking. A response does not need citations "
            "or formal evidence to be cogent."
        ),
        "short": (
            "Which response advances the more cogent argument? Consider whether its reasons are acceptable, "
            "relevant, and sufficient. Judge argument quality, not whether this particular OP would be persuaded."
        ),
        "question": "Which response advances the more cogent argument?"
    },
    "effectiveness": {
        "title": "EXPECTED RHETORICAL EFFECTIVENESS — LIKELIHOOD OF CHANGING THIS OP'S VIEW",
        "full": (
            "Predict which response is more likely to change this particular OP's stated view. Use the original "
            "post to understand the OP's stated position, reasons, concerns, values, priorities, and other context "
            "they provide. You may make reasonable inferences from this information when predicting what might "
            "persuade them. In many cases, the OP will not explicitly state everything that matters, so use your "
            "best judgment based on the context available in the post. Do not assume unsupported personal "
            "characteristics or use information from outside the material shown. This is an ex ante judgment: "
            "predict which response is more likely to change the OP's view without knowing what actually happened "
            "afterward. If useful, you may consider rhetorical features such as credibility, emotional appeal, "
            "clarity, appropriateness, and arrangement. These are guidance, not a checklist, and they do not define "
            "the answer. Other aspects of the response may also matter. Cogency may contribute to rhetorical "
            "effectiveness, but do not simply repeat your cogency judgment. Do not judge whether you would be persuaded."
        ),
        "short": (
            "Which response is more likely to change this particular OP's stated view? Use the context available "
            "in the OP's post, including reasonable inferences from what the OP says, and make your best prediction. "
            "Optional cues include credibility, emotional appeal, clarity, appropriateness, and arrangement; "
            "these are guidance, not separate criteria."
        ),
        "question": "Which response is more likely to change this particular OP's stated view?"
    },
    "evidence": {
        "title": "EVIDENTIAL SUPPORT",
        "full": (
            "Evidence is concrete supporting information offered for a claim. It may include facts, data, statistics, "
            "research findings, documented real-world examples, or information attributed to an identifiable source. "
            "A bare assertion, opinion, value judgment, rhetorical question, insult, or simple restatement of a conclusion "
            "does not count as evidence by itself. First, indicate independently whether each response provides any "
            "concrete supporting information beyond a bare assertion, opinion, or restatement. If both responses provide "
            "evidence, compare their evidential support. Consider how specific and concrete the evidence is, how well it "
            "supports the claim being made, and, where relevant, whether the response identifies the source or basis of "
            "the evidence. Do not open links, search for sources, or independently fact-check claims. A citation or "
            "institutional source is not automatically stronger evidence."
        ),
        "short": (
            "First decide independently whether A and B provide concrete information in support of their arguments. "
            "If both do, compare how specific and concrete the evidence is, how well it supports the claim being made, "
            "and, where relevant, whether the response identifies the source or basis of the evidence. Do not open links "
            "or independently verify the claims."
        ),
        "presence_question": "Which responses provide evidence in support of their argument? Check all that apply.",
        "comparison_question": "Which response provides stronger evidential support for its argument?"
    },
    "unusual_responses": (
        "Some randomly sampled responses may be questions, jokes, insults, simple agreement or disagreement, "
        "bare assertions, or otherwise provide little argumentative content. This is expected. At the bottom of "
        "every annotation screen there are optional checkboxes labeled A: Nonargumentative, B: Nonargumentative, "
        "A: Unreadable / truncated, and B: Unreadable / truncated. Check Nonargumentative only when that response "
        "clearly provides no supporting reason or argumentative contribution. Check Unreadable / truncated only "
        "when the text itself cannot reasonably be evaluated. A flag does not replace the annotation: make the "
        "requested judgments as best you can."
    ),
    "examples": [
        {
            "title": "Example 1 — Cogency",
            "post": "CMV: Campus dining does not need to stay open after 8 p.m. because students can simply eat earlier.",
            "a": "Some required evening classes end after 8 p.m. Students on meal plans may therefore lose access to food they have already paid for.",
            "b": "Campus dining should stay open later because 8 p.m. is too early, and many students probably prefer eating later anyway.",
            "lesson": "Response A may be more cogent because its reasons are directly relevant to the conclusion and provide clearer support for it. The example illustrates the cogency criteria; real annotations may be less clear-cut."
        },
        {
            "title": "Example 2 — Expected Rhetorical Effectiveness",
            "post": "CMV: I do not use two-factor authentication because I do not want to enter another code every time I log in.",
            "a": "Two-factor authentication adds an extra step, but the added security is worth the inconvenience because it makes accounts much harder to compromise.",
            "b": "Many services can remember a trusted device, so you may not need a code every time. That directly addresses the inconvenience you are worried about.",
            "lesson": "Response B may be more likely to change this particular OP's stated view because it directly addresses the concern the OP identified. In many real annotations, you will need to make your best prediction from the contextual information in the OP's post. The task is not which response would persuade people in general."
        },
        {
            "title": "Example 3 — Evidential Support: evidence present versus absent",
            "post": "CMV: My local bus service seems reliable enough.",
            "a": "The bus system is unreliable because everyone who uses Route 18 knows that buses are late all the time.",
            "b": "The city's 2025 transit report says Route 18 arrived more than five minutes late on 27% of recorded trips.",
            "lesson": "Response A makes an empirical claim but provides no concrete supporting information, so its evidence box would remain unchecked. Response B provides a specific statistic and identifies its source, so its evidence box would be checked. You do not need to verify the report."
        },
        {
            "title": "Example 4 — Evidential Support: both responses contain evidence",
            "post": "CMV: Route 18 is reliable enough that occasional delays are not a real problem.",
            "a": "A local group called Transit Riders for Better Service surveyed 52 people at its 'Fix Our Buses' meeting. Forty-six said Route 18 is frequently late.",
            "b": "The city transit agency's automatic vehicle records show that Route 18 arrived more than five minutes late on 29% of 8,412 recorded trips during the last six months.",
            "lesson": "Both responses provide evidence, so both evidence boxes would be checked. Response A gives concrete survey results, but the respondents are a small, self-selected group attending a meeting specifically about bus problems. Response B may provide stronger evidential support for a general claim about Route 18's reliability because it directly summarizes thousands of recorded trips over an extended period. The task is to judge how well the evidence presented supports the claim, not to verify either source externally."
        }
    ],
    "references": [
        "Johnson & Blair (2006); Blair (2012); Wachsmuth et al. (2017) for cogency and argument-quality structure.",
        "Wachsmuth et al. (2017) and recipient-sensitive persuasion literature for rhetorical-effectiveness guidance.",
        "Wachsmuth et al. (2017) and related evidence-annotation work for evidential-support guidance."
    ]
}

# Save a machine-readable copy for the research workflow.
with open(PROTOCOL_JSON, "w", encoding="utf-8") as f:
    json.dump(PROTOCOL, f, ensure_ascii=False, indent=2)


def _esc(x): return html.escape(str(x))


def _panel(title, body, accent="#0d2b4a"):
    return f"""
    <div style='border:1px solid #9aa6b2;border-left:6px solid {accent};border-radius:6px;
                padding:14px 16px;margin:12px 0;background:white'>
      <div style='font:800 16px sans-serif;color:{accent};margin-bottom:8px'>{_esc(title)}</div>
      <div style='font:14.5px/1.62 sans-serif;color:#111'>{body}</div>
    </div>"""


scale_html = "<ul>" + "".join(
    f"<li><b>{_esc(k)}:</b> {_esc(v)}</li>" for k,v in PROTOCOL["scale"].items()
) + "</ul>"

# Overview, then the four-way scale, then the three construct definitions.
display(HTML(_panel("OVERVIEW", _esc(PROTOCOL["overview"]))))
display(HTML(_panel("FOUR-WAY COMPARATIVE SCALE", _esc(PROTOCOL["scale_intro"]) + scale_html)))

for key in ("cogency", "effectiveness", "evidence"):
    q = PROTOCOL[key]
    body = f"<div>{_esc(q['full'])}</div>"
    if key == "evidence":
        body += f"<div style='margin-top:10px'><b>First:</b> {_esc(q['presence_question'])}</div>"
        body += f"<div style='margin-top:6px'><b>If both are checked:</b> {_esc(q['comparison_question'])}</div>"
    else:
        body += f"<div style='margin-top:10px'><b>Decision question:</b> {_esc(q['question'])}</div>"
    display(HTML(_panel(q["title"], body)))

# The response-flag guidance comes after the three annotation definitions.
display(HTML(_panel("RESPONSES THAT ARE NOT CLEAR ARGUMENTS", _esc(PROTOCOL["unusual_responses"]))))

display(Markdown("## Worked examples"))
for ex in PROTOCOL["examples"]:
    body = (
        f"<b>Original post:</b> {_esc(ex['post'])}<br><br>"
        f"<b>Response A:</b> {_esc(ex['a'])}<br><br>"
        f"<b>Response B:</b> {_esc(ex['b'])}<br><br>"
        f"<b>Illustration:</b> {_esc(ex['lesson'])}"
    )
    display(HTML(_panel(ex["title"], body, accent="#355d3a")))


# 3. Calibration round — first 5 OPs (75 pair assignments)

Every annotator completes the **same fixed first 5 OPs** from the shared sample. Each OP has 6 sampled responses, so all 15 within-OP response pairs are shown, for a total of **75 pair assignments**.

Complete this section **independently**. Do not discuss individual items or compare answers with another annotator while doing the calibration round. Judge each displayed pair on its own; do not change earlier answers simply to make your responses form a perfectly consistent ranking.

When all 75 pairs are complete, run the **Finalize and submit the calibration baseline** cell below. Submit that CSV to the research team for inter-rater agreement review, then **stop and wait for the research team before beginning the main annotation round**.

The purpose of calibration is to identify systematic differences in how the instructions are being interpreted. There are no hidden “correct” answers for genuinely subjective cases, and the goal is not perfect agreement.


In [ ]:
# ============================================================
# REUSABLE ANNOTATION UI
# ============================================================

# ---------- Reddit-ish text rendering ----------
_URL = re.compile(r'((?:https?://|www\.)[^\s<>()\[\]]+[^\s<>()\[\].,;:!?\'\"])')

def _short_url(u):
    m = re.match(r'(?:https?://)?(?:www\.)?([^/\s]+)', u)
    return m.group(1) if m else u

def _inline(s):
    store = []
    def keep(m):
        store.append(m.group(1)); return f"\x00{len(store)-1}\x00"
    s = re.sub(r'`([^`\n]+)`', keep, s)
    s = re.sub(r'\[([^\]\n]+)\]\(\s*(\S+?)(?:\s+&quot;[^)]*&quot;)?\s*\)',
               lambda m: (f"<span class='cmv-link'>{m.group(1)}</span>"
                          f"<span class='cmv-url'> [{_short_url(m.group(2))}]</span>"), s)
    s = _URL.sub(lambda m: f"<span class='cmv-link'>{m.group(1)}</span>", s)
    s = re.sub(r'\*\*(.+?)\*\*', r'<b>\1</b>', s)
    s = re.sub(r'(?<![\w*])\*([^*\n]+)\*(?![\w*])', r'<i>\1</i>', s)
    for i, c in enumerate(store):
        s = s.replace(f"\x00{i}\x00", f"<code>{c}</code>")
    return s

def _esc_inline(txt):
    return _inline(html.escape(str(txt))).replace("\n", "<br>")

def render_text(src):
    s = html.unescape(str(src or "")).replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r'\n{3,}', '\n\n', s)
    parts = []
    for para in s.split("\n\n"):
        para = para.strip()
        if not para:
            continue
        if all(line.lstrip().startswith(">") for line in para.splitlines()):
            q = "\n".join(line.lstrip()[1:].lstrip() for line in para.splitlines())
            parts.append(f"<blockquote class='cmv-bq'>{_esc_inline(q)}</blockquote>")
        else:
            parts.append(f"<p class='cmv-p'>{_esc_inline(para)}</p>")
    return "".join(parts) or "<p><i>(empty)</i></p>"

# ---------- Shared style ----------
# Keep the visual design aligned with the original student UI: a white
# decision card, light-blue scale segments, and a four-position slider.
ACCENT = "#0d2b4a"
COL_W  = 624
SEG_W, SEG_GAP = 148, 4
CENTRES = [78, 234, 390, 546]
HANDLE_R = 13
SL_LEFT = CENTRES[0]
SL_W = CENTRES[-1] - CENTRES[0]
STOPS = [0, 100, 200, 300]
MID = 150

# Explicit foreground/background colors are used so the UI remains readable
# under both light and dark notebook themes.
display(HTML(f"""
<style>
.cmv-surface {{ background:#f3f5f7 !important; border-radius:8px !important;
  padding:12px 10px 18px 10px !important; }}
.cmv-ctl {{ background:#ffffff !important; border:1px solid #8d9aa8 !important;
  border-radius:7px !important; padding:10px 0 8px 0 !important; }}
.cmv-white {{ background:#ffffff !important; border:1px solid #8d9aa8 !important;
  border-radius:7px !important; padding:10px 12px !important; }}
.cmv-check, .cmv-check label, .cmv-check span {{ color:#111111 !important; }}
.cmv-slider .widget-readout {{ display:none !important; }}
.cmv-slider .ui-slider {{ height:8px !important; border-radius:4px !important;
  background:#d3dce6 !important; border:1px solid #8d9aa8 !important; }}
.cmv-slider .ui-slider .ui-slider-range {{ background:transparent !important; }}
.cmv-slider .ui-slider .ui-slider-handle {{
  width:{2*HANDLE_R}px !important; height:{2*HANDLE_R}px !important;
  border-radius:50% !important; background:#0d2b4a !important;
  border:3px solid #ffffff !important;
  box-shadow:0 1px 6px rgba(0,0,0,.45) !important;
  margin-left:-{HANDLE_R}px !important; cursor:grab !important; }}
.cmv-slider .noUi-target {{ height:8px !important; border-radius:4px !important;
  background:#d3dce6 !important; border:1px solid #8d9aa8 !important;
  box-shadow:none !important; }}
.cmv-slider .noUi-connects, .cmv-slider .noUi-connect {{ background:transparent !important; }}
.cmv-slider .noUi-handle {{
  width:{2*HANDLE_R}px !important; height:{2*HANDLE_R}px !important;
  border-radius:50% !important; background:#0d2b4a !important;
  border:3px solid #ffffff !important;
  box-shadow:0 1px 6px rgba(0,0,0,.45) !important;
  right:-{HANDLE_R}px !important; cursor:grab !important; }}
.cmv-slider .noUi-handle:before, .cmv-slider .noUi-handle:after {{ display:none !important; }}
.cmv-mid .ui-slider .ui-slider-handle, .cmv-mid .noUi-handle {{
  background:#ffffff !important; border:3px solid #6b7580 !important;
  box-shadow:0 1px 4px rgba(0,0,0,.22) !important; }}
.cmv-body {{ font:15px/1.62 Georgia,serif; color:#000000 !important; }}
.cmv-body p.cmv-p {{ margin:0 0 10px 0; color:#000000 !important; }}
.cmv-body blockquote.cmv-bq {{ margin:8px 0; padding:2px 0 2px 12px;
  border-left:3px solid #8d9aa8; color:#26303a !important; }}
.cmv-body code {{ background:#eceff2; border-radius:3px; padding:1px 4px;
  font:13px Menlo,Consolas,monospace; color:#000000 !important; }}
.cmv-link {{ color:#0b3d7a !important; text-decoration:underline; }}
.cmv-url {{ color:#4a5561 !important; font:11.5px sans-serif; }}
</style>
"""))

CHOICES = ["Strong A", "Weak A", "Weak B", "Strong B"]
SIGNED = {"Strong A": -2, "Weak A": -1, "Weak B": 1, "Strong B": 2}
MARGIN = {"Strong A": 2, "Weak A": 1, "Weak B": 1, "Strong B": 2}
MAGNITUDE = {"Strong A": "strong", "Weak A": "weak", "Weak B": "weak", "Strong B": "strong"}
FILL = {
    ("strong", False): ("#b8cfea", "#08243f"),
    ("strong", True):  ("#0d2b4a", "#ffffff"),
    ("weak", False):   ("#dce9f7", "#0d2b4a"),
    ("weak", True):    ("#2c6bab", "#ffffff"),
}

class SegSlider:
    """Four-position slider matching the original annotation UI."""
    def __init__(self, opts, on_change=None):
        self.opts, self.value, self.on_change = list(opts), None, on_change
        self._silent = False
        self.labels = widgets.HTML(layout=widgets.Layout(margin="0"))
        self.notches = widgets.HTML(
            f"<div style='position:relative;width:{COL_W}px;height:9px'>" +
            "".join(
                f"<div style='position:absolute;left:{x-1}px;top:0;width:2px;height:9px;"
                f"background:#0d2b4a;border-radius:1px'></div>" for x in CENTRES
            ) + "</div>",
            layout=widgets.Layout(margin="5px 0 1px 0")
        )
        self.slider = widgets.IntSlider(
            min=0, max=300, step=1, value=MID, readout=False,
            continuous_update=False,
            layout=widgets.Layout(width=f"{SL_W}px", margin=f"0 0 0 {SL_LEFT}px")
        )
        self.slider.add_class("cmv-slider")
        self.slider.add_class("cmv-mid")
        self.slider.observe(self._slide, names="value")
        self.status = widgets.HTML(layout=widgets.Layout(margin="6px 0 0 0"))
        self.widget = widgets.VBox(
            [self.labels, self.notches, self.slider, self.status],
            layout=widgets.Layout(width=f"{COL_W + 2}px", align_items="flex-start")
        )
        self.widget.add_class("cmv-ctl")
        self._render()

    def _render(self):
        cells = []
        for o in self.opts:
            bg, fg = FILL[(MAGNITUDE[o], o == self.value)]
            bd = "2px solid #0d2b4a" if o == self.value else "1px solid #8d9aa8"
            wt = 800 if o == self.value else 700
            cells.append(
                f"<div style='box-sizing:border-box;width:{SEG_W}px;height:36px;"
                f"margin:0 {SEG_GAP}px;border:{bd};border-radius:6px;background:{bg};"
                f"color:{fg};font:{wt} 13px/34px sans-serif;text-align:center'>{o}</div>"
            )
        self.labels.value = f"<div style='display:flex;width:{COL_W}px'>{''.join(cells)}</div>"
        if self.value is None:
            self.status.value = (
                f"<div style='box-sizing:border-box;width:{COL_W}px;text-align:center;"
                f"background:#e4e7ea;border-radius:4px;padding:5px 0;"
                f"font:700 12.5px sans-serif;color:#1a1a1a'>"
                f"Not answered &mdash; nudge the handle left or right</div>"
            )
        else:
            self.status.value = (
                f"<div style='box-sizing:border-box;width:{COL_W}px;text-align:center;"
                f"background:#dce9f7;border-radius:4px;padding:5px 0;"
                f"font:800 13px sans-serif;color:#0d2b4a'>{self.value}</div>"
            )

    def _slide(self, ch):
        if self._silent:
            return
        v = int(ch["new"])
        if v == MID:
            if self.value is not None:
                self._push(STOPS[self.opts.index(self.value)])
            return
        i = min(range(4), key=lambda k: abs(v - STOPS[k]))
        self.set(self.opts[i])
        if self.on_change:
            self.on_change()

    def _push(self, v):
        self._silent = True
        try:
            self.slider.value = v
        finally:
            self._silent = False

    def set(self, o):
        self.value = o if o in self.opts else None
        if self.value is None:
            self._push(MID)
            self.slider.add_class("cmv-mid")
        else:
            self._push(STOPS[self.opts.index(self.value)])
            self.slider.remove_class("cmv-mid")
        self._render()

    def reset(self):
        self.set(None)


def readbox(title, body_html, maxh=320):
    return (
        f"<div style='border:1px solid #8d9aa8;border-radius:6px;background:#ffffff;padding:11px 14px'>"
        f"<div style='font:800 12px sans-serif;color:{ACCENT};letter-spacing:.08em;margin-bottom:8px'>"
        f"{html.escape(title)}</div>"
        f"<div class='cmv-body' style='max-height:{maxh}px;overflow-y:auto;color:#000000'>{body_html}</div></div>"
    )


SCALE_UI_TEXT = (
    "Weak = one response is somewhat better or you only slightly prefer it. "
    "Strong = one response is clearly better. No ties: if they seem nearly equal, "
    "choose the side you lean toward and use Weak."
)


def qbox(title, reminder, decision=None, extra=None, show_scale=False):
    bits = [
        f"<div style='font:800 15px sans-serif;color:{ACCENT};letter-spacing:.035em'>"
        f"{html.escape(title)}</div>",
        f"<div style='font:13.5px/1.55 sans-serif;color:#111111;margin-top:7px'>"
        f"{html.escape(reminder)}</div>"
    ]
    if extra:
        bits.append(
            f"<div style='font:13px/1.5 sans-serif;color:#35506a;margin-top:7px'>"
            f"{html.escape(extra)}</div>"
        )
    if decision:
        bits.append(
            f"<div style='margin-top:10px;padding-top:9px;border-top:2px solid {ACCENT};"
            f"font:800 15px/1.45 sans-serif;color:#000000'>{html.escape(decision)}</div>"
        )
    if show_scale:
        bits.append(
            f"<div style='margin-top:8px;font:13px/1.5 sans-serif;color:#1a1a1a'>"
            f"{html.escape(SCALE_UI_TEXT)}</div>"
        )
    return widgets.HTML(
        f"<div style='box-sizing:border-box;width:{COL_W+2}px;background:#ffffff;"
        f"border:1px solid #8d9aa8;border-left:6px solid {ACCENT};border-radius:5px;"
        f"padding:13px 15px;margin:18px 0 7px 0'>" + "".join(bits) + "</div>"
    )


def white_box(children, width=COL_W):
    box = widgets.VBox(children, layout=widgets.Layout(width=f"{width}px"))
    box.add_class("cmv-white")
    return box


def _as_bool(v):
    if pd.isna(v): return False
    if isinstance(v, str): return v.strip().lower() in ("true", "1", "yes")
    return bool(v)


def launch_annotation_ui(tasks, phase_label, review_mode=False):
    tasks = list(tasks)
    assert tasks, "No tasks supplied."

    w_prog = widgets.HTML()
    w_op = widgets.HTML()
    w_a = widgets.HTML(layout=widgets.Layout(width="49.3%"))
    w_b = widgets.HTML(layout=widgets.Layout(width="49.3%"))
    w_hide = widgets.Checkbox(value=False, indent=False, description="Collapse original post text")
    w_hide.add_class("cmv-check")
    w_msg = widgets.HTML()

    def msg(text, kind="info"):
        bg, fg = {"err": ("#fbe3e3", "#8f1420"), "ok": ("#ddf0e4", "#0b3d22"),
                  "info": ("#e9edf1", "#1a1a1a")}[kind]
        w_msg.value = (f"<div style='width:{COL_W}px;background:{bg};color:{fg};border-radius:4px;"
                       f"padding:8px 12px;font:700 13px sans-serif'>{text}</div>" if text else "")

    cog = SegSlider(CHOICES, on_change=lambda: msg(""))
    eff = SegSlider(CHOICES, on_change=lambda: msg(""))
    evi = SegSlider(CHOICES, on_change=lambda: msg(""))

    cog_panel = widgets.VBox([
        qbox(PROTOCOL["cogency"]["title"], PROTOCOL["cogency"]["short"],
             PROTOCOL["cogency"]["question"], show_scale=True),
        cog.widget
    ], layout=widgets.Layout(width=f"{COL_W+2}px"))

    eff_panel = widgets.VBox([
        qbox(PROTOCOL["effectiveness"]["title"], PROTOCOL["effectiveness"]["short"],
             PROTOCOL["effectiveness"]["question"], show_scale=True),
        eff.widget
    ], layout=widgets.Layout(width=f"{COL_W+2}px"))

    ev_a = widgets.Checkbox(value=False, indent=False, description="Response A provides evidence",
                            layout=widgets.Layout(width="292px"))
    ev_b = widgets.Checkbox(value=False, indent=False, description="Response B provides evidence",
                            layout=widgets.Layout(width="292px"))
    ev_a.add_class("cmv-check")
    ev_b.add_class("cmv-check")
    ev_presence_controls = widgets.HBox(
        [ev_a, ev_b], layout=widgets.Layout(width=f"{COL_W-24}px", justify_content="space-between")
    )
    ev_presence_box = white_box([ev_presence_controls], width=COL_W)
    ev_compare_box = widgets.VBox([], layout=widgets.Layout(width=f"{COL_W}px"))

    ev_head = qbox(PROTOCOL["evidence"]["title"], PROTOCOL["evidence"]["short"],
                   PROTOCOL["evidence"]["presence_question"])

    def update_evidence_visibility(*_):
        if ev_a.value and ev_b.value:
            ev_compare_box.children = [
                qbox("EVIDENTIAL SUPPORT — COMPARISON",
                     "Both responses provide evidence. Compare the evidence itself, not the overall writing or argument.",
                     PROTOCOL["evidence"]["comparison_question"], show_scale=True),
                evi.widget
            ]
        else:
            evi.reset()
            ev_compare_box.children = []
        msg("")

    ev_a.observe(update_evidence_visibility, names="value")
    ev_b.observe(update_evidence_visibility, names="value")

    evidence_panel = widgets.VBox([
        ev_head,
        ev_presence_box,
        ev_compare_box
    ], layout=widgets.Layout(width=f"{COL_W+2}px"))

    # Separate item-level flags preserve naturally occurring floor/sanity-check cases.
    flag_a_nonarg = widgets.Checkbox(value=False, indent=False, description="A: Nonargumentative")
    flag_b_nonarg = widgets.Checkbox(value=False, indent=False, description="B: Nonargumentative")
    flag_a_bad = widgets.Checkbox(value=False, indent=False, description="A: Unreadable / truncated")
    flag_b_bad = widgets.Checkbox(value=False, indent=False, description="B: Unreadable / truncated")
    for _w in (flag_a_nonarg, flag_b_nonarg, flag_a_bad, flag_b_bad):
        _w.add_class("cmv-check")
    flags = white_box([
        widgets.HTML(f"<div style='font:700 13px sans-serif;color:{ACCENT};margin-bottom:7px'>Optional response flags — check only clear cases</div>"),
        widgets.HBox([flag_a_nonarg, flag_b_nonarg], layout=widgets.Layout(width=f"{COL_W-24}px")),
        widgets.HBox([flag_a_bad, flag_b_bad], layout=widgets.Layout(width=f"{COL_W-24}px")),
    ], width=COL_W)

    notes = widgets.Textarea(placeholder="Optional note", layout=widgets.Layout(width=f"{COL_W}px", height="54px"))
    b_back = widgets.Button(description="← Back", layout=widgets.Layout(width="110px", height="42px"))
    b_submit = widgets.Button(description="Submit →", button_style="success", layout=widgets.Layout(width="180px", height="42px"))
    b_skip = widgets.Button(description="Skip for now", layout=widgets.Layout(width="140px", height="42px"))

    # In review mode, a post selector makes deliberate revision easier without encouraging routine backtracking.
    post_selector = None
    if review_mode:
        post_options = []
        seen = set()
        for t in tasks:
            if t["post_id"] not in seen:
                seen.add(t["post_id"])
                title = OP[t["post_id"]][0]
                post_options.append((f"OP {t['sample_post_order']+1}: {title[:70]}", t["post_id"]))
        post_selector = widgets.Dropdown(options=post_options, description="Jump to OP:",
                                         layout=widgets.Layout(width=f"{COL_W}px"))

    questions = widgets.VBox([], layout=widgets.Layout(width="100%", align_items="center"))
    footer_items = [flags, notes,
                    widgets.HBox([b_back, b_submit, b_skip], layout=widgets.Layout(width=f"{COL_W}px", margin="8px 0")),
                    w_msg]
    footer = widgets.VBox(footer_items, layout=widgets.Layout(width="100%", align_items="center"))

    ui_items = [w_prog]
    if post_selector is not None:
        ui_items.append(widgets.HBox([post_selector], layout=widgets.Layout(width="100%", justify_content="center")))
    ui_items += [widgets.HBox([w_hide], layout=widgets.Layout(width="100%", justify_content="center")),
                 w_op, widgets.HTML("<div style='height:8px'></div>"),
                 widgets.HBox([w_a, w_b], layout=widgets.Layout(width="100%", justify_content="space-between")),
                 questions, footer]
    UI = widgets.VBox(ui_items, layout=widgets.Layout(width="100%"))
    UI.add_class("cmv-surface")

    class State:
        idx = 0
        t0 = time.time()
        last_post = None

    def first_target():
        if review_mode:
            return 0
        for i, t in enumerate(tasks):
            if t["pair_key"] not in RECORDS:
                return i
        return max(0, len(tasks)-1)

    State.idx = first_target()

    def render_post():
        t = tasks[State.idx]
        title, body = OP[t["post_id"]]
        head = f"<b style='font-size:17px'>{html.escape(title)}</b>"
        w_op.value = readbox("THE ORIGINAL POST", head + (" <i>(collapsed)</i>" if w_hide.value else "<br><br>" + render_text(body)), 310)

    def load_previous(t):
        cog.reset(); eff.reset(); evi.reset()
        ev_a.value = False; ev_b.value = False
        flag_a_nonarg.value = flag_b_nonarg.value = False
        flag_a_bad.value = flag_b_bad.value = False
        notes.value = ""
        prev = RECORDS.get(t["pair_key"])
        if prev:
            cog.set(prev.get("cogency_choice"))
            eff.set(prev.get("effect_choice"))
            ev_a.value = _as_bool(prev.get("evidence_a_present"))
            ev_b.value = _as_bool(prev.get("evidence_b_present"))
            update_evidence_visibility()
            if ev_a.value and ev_b.value:
                evi.set(prev.get("evidence_choice"))
            flag_a_nonarg.value = _as_bool(prev.get("a_nonargumentative"))
            flag_b_nonarg.value = _as_bool(prev.get("b_nonargumentative"))
            flag_a_bad.value = _as_bool(prev.get("a_unreadable"))
            flag_b_bad.value = _as_bool(prev.get("b_unreadable"))
            notes.value = "" if pd.isna(prev.get("notes")) else str(prev.get("notes"))
            b_submit.description = "Update →"
            msg("Previously answered. Updating will preserve the earlier version in the history file.", "info")
        else:
            update_evidence_visibility()
            b_submit.description = "Submit →"
            msg("")

    def paint():
        t = tasks[State.idx]
        pid = t["post_id"]
        if pid != State.last_post:
            w_hide.value = False
            State.last_post = pid

        n_done = completed(tasks)
        phase_name = "CALIBRATION" if t["phase"] == "calibration" else "MAIN ANNOTATION"
        if review_mode:
            phase_name += " — OPTIONAL REVIEW/REVISION"
        w_prog.value = (
            f"<div style='background:{ACCENT};color:white;border-radius:5px;padding:9px 14px;"
            f"font:700 14px sans-serif;text-align:center'>{phase_name} &nbsp;·&nbsp; "
            f"OP {t['phase_post_order']+1} of {len({x['post_id'] for x in tasks})} &nbsp;·&nbsp; "
            f"pair {t['pair_in_post']+1} of 15 &nbsp;·&nbsp; {n_done}/{len(tasks)} saved</div>"
        )
        render_post()
        w_a.value = readbox("RESPONSE A", render_text(TEXT[t["response_id_a"]]), 340)
        w_b.value = readbox("RESPONSE B", render_text(TEXT[t["response_id_b"]]), 340)
        questions.children = [cog_panel, eff_panel, evidence_panel]
        load_previous(t)
        if post_selector is not None:
            post_selector.value = pid
        State.t0 = time.time()

    def resolve(choice, t):
        rid = t["response_id_a"] if choice.endswith("A") else t["response_id_b"]
        return rid, MARGIN[choice], SIGNED[choice]

    def submit(_):
        t = tasks[State.idx]
        if cog.value is None:
            return msg("Please answer COGENCY.", "err")
        if eff.value is None:
            return msg("Please answer EXPECTED RHETORICAL EFFECTIVENESS.", "err")
        if ev_a.value and ev_b.value and evi.value is None:
            return msg("Both responses are marked as providing evidence. Please compare their EVIDENTIAL SUPPORT.", "err")

        now = datetime.now(timezone.utc).isoformat(timespec="seconds")
        old = RECORDS.get(t["pair_key"])
        cog_rid, cog_m, cog_s = resolve(cog.value, t)
        eff_rid, eff_m, eff_s = resolve(eff.value, t)

        if ev_a.value and ev_b.value:
            ev_rid, ev_m, ev_s = resolve(evi.value, t)
            ev_choice = evi.value
            presence_pattern = "both"
        elif ev_a.value:
            ev_rid = ev_m = ev_s = ev_choice = pd.NA
            presence_pattern = "A_only"
        elif ev_b.value:
            ev_rid = ev_m = ev_s = ev_choice = pd.NA
            presence_pattern = "B_only"
        else:
            ev_rid = ev_m = ev_s = ev_choice = pd.NA
            presence_pattern = "neither"

        elapsed = round(time.time() - State.t0, 1)
        first_time = now if old is None or pd.isna(old.get("first_submitted_utc")) else old.get("first_submitted_utc")
        old_seconds = 0.0 if old is None or pd.isna(old.get("seconds_on_pair_total")) else float(old.get("seconds_on_pair_total"))
        old_revs = -1 if old is None else int(0 if pd.isna(old.get("revision_count")) else old.get("revision_count"))

        rec = {
            "pair_key": t["pair_key"], "annotator_id": ANNOTATOR_ID,
            "instruction_version": PROTOCOL_VERSION, "phase_of_last_edit": phase_label,
            "post_id": t["post_id"], "sample_post_order": t["sample_post_order"],
            "phase_post_order": t["phase_post_order"], "pair_in_post": t["pair_in_post"],
            "response_id_a": t["response_id_a"], "response_id_b": t["response_id_b"],
            "ab_flipped": t["ab_flipped"], "task_order": t["task_order"],
            "cogency_choice": cog.value, "cogency_winner_response_id": cog_rid,
            "cogency_margin": cog_m, "cogency_signed": cog_s,
            "effect_choice": eff.value, "effect_winner_response_id": eff_rid,
            "effect_margin": eff_m, "effect_signed": eff_s,
            "evidence_a_present": bool(ev_a.value), "evidence_b_present": bool(ev_b.value),
            "evidence_presence_pattern": presence_pattern,
            "evidence_choice": ev_choice, "evidence_winner_response_id": ev_rid,
            "evidence_margin": ev_m, "evidence_signed": ev_s,
            "a_nonargumentative": bool(flag_a_nonarg.value), "b_nonargumentative": bool(flag_b_nonarg.value),
            "a_unreadable": bool(flag_a_bad.value), "b_unreadable": bool(flag_b_bad.value),
            "notes": notes.value.strip(), "seconds_on_pair_total": round(old_seconds + elapsed, 1),
            "revision_count": max(0, old_revs + 1),
            "first_submitted_utc": first_time, "last_modified_utc": now,
        }
        RECORDS[t["pair_key"]] = rec

        event = {"event_index": len(HISTORY) + 1,
                 "event_type": "initial_submission" if old is None else "update",
                 "event_phase": phase_label, "event_utc": now, **rec}
        HISTORY.append(event)
        save_records()

        if State.idx < len(tasks)-1:
            State.idx += 1
            paint()
        else:
            msg("This section is complete. Your answers are saved.", "ok")

    def back(_):
        State.idx = max(0, State.idx-1)
        paint()

    def skip(_):
        State.idx = min(len(tasks)-1, State.idx+1)
        paint()

    def jump(change):
        if not review_mode or change.get("name") != "value":
            return
        pid = change["new"]
        for i,t in enumerate(tasks):
            if t["post_id"] == pid:
                State.idx = i
                State.last_post = None
                paint()
                break

    b_submit.on_click(submit)
    b_back.on_click(back)
    b_skip.on_click(skip)
    w_hide.observe(lambda ch: render_post(), names="value")
    if post_selector is not None:
        post_selector.observe(jump, names="value")

    display(UI)
    paint()
    return UI

# Launch the calibration UI.
CALIBRATION_UI = launch_annotation_ui(CAL_TASKS, phase_label="calibration_initial", review_mode=False)


## 3.1 Finalize and submit the calibration baseline

Run the next cell **only after all 75 calibration pairs are complete**. It creates `calibration_baseline_<annotator>.csv`, which contains your calibration answers before any calibration discussion.

You will find this file inside the **`annotations/` folder in the same project directory as this notebook**. The cell will also print the full file location so you can find it easily.

Submit `calibration_baseline_<annotator>.csv` to the research team for inter-rater agreement review, then stop and wait for instructions before continuing.


In [ ]:
# ============================================================
# FREEZE THE PRE-DISCUSSION CALIBRATION BASELINE
# ============================================================

assert completed(CAL_TASKS) == len(CAL_TASKS), (
    f"Calibration is not complete: {completed(CAL_TASKS)} / {len(CAL_TASKS)} pairs saved."
)

if CAL_BASELINE_CSV.exists():
    print(f"Calibration baseline file already exists:\n{CAL_BASELINE_CSV}")
else:
    baseline = current_calibration_df()
    assert len(baseline) == 75
    _atomic_write(baseline, CAL_BASELINE_CSV)
    print(f"Calibration baseline created:\n{CAL_BASELINE_CSV}")

# Always refresh the current calibration snapshot.
_atomic_write(current_calibration_df(), CAL_CURRENT_CSV)
print(f"Current calibration file:\n{CAL_CURRENT_CSV}")
print("\nFind the baseline file in the annotations/ folder, submit it for inter-rater agreement review, then stop until the research team responds.")


# 4. Calibration revision (Optional)

Use this section **only if the research team asks you to revisit calibration judgments after reviewing agreement and clarifying the instructions**.

The review interface lets you return to any of the 75 calibration pairs and update a judgment when the clarified instructions change how you interpret the task. Any changes are recorded as revisions, so the research team can distinguish your initial calibration judgments from later updates.

Revisions should follow clarified **general instructions**. Do not change an answer merely to match another annotator on a subjective case, and do not treat the calibration discussion as an answer key.


In [ ]:
# ============================================================
# OPTIONAL CALIBRATION REVIEW / REVISION UI
# ============================================================
assert CAL_BASELINE_CSV.exists(), "Finalize the calibration baseline before using this revision section."
CALIBRATION_REVISION_UI = launch_annotation_ui(
    CAL_TASKS,
    phase_label="calibration_revision",
    review_mode=True,
)


# 5. Main annotation round — remaining 115 OPs (1,725 pair assignments)

Begin this section **only after the research team has reviewed the calibration round and told you to proceed**.

Apply the same finalized definitions and decision rules used during calibration. Work independently and do not discuss specific annotation items with other annotators.

The remaining 115 OPs are shown in a stable annotator-specific order. All 15 response pairs from an OP appear consecutively. The notebook saves after every submitted pair, so you may stop and resume later. When practical, finish all 15 pairs for the current OP before taking a break.

As during calibration, judge each displayed pair on its own. Do not revise an earlier answer merely to make the full set of comparisons form a perfectly consistent ranking.


In [ ]:
# ============================================================
# MAIN ANNOTATION UI
# ============================================================
assert CAL_BASELINE_CSV.exists(), (
    "The calibration baseline has not been finalized. Complete Section 3.1 before starting the main round."
)
MAIN_UI = launch_annotation_ui(MAIN_TASKS, phase_label="main_annotation", review_mode=False)


# 6. What to submit when you are finished

After you have completed the calibration round and all 115 main-round OPs, submit the following **three CSV files** from the `annotation/output/` folder:

1. `calibration_baseline_<annotator>.csv` — your original calibration-round answers used for inter-rater agreement.
2. `annotations_<annotator>.csv` — your final current annotations for all 120 OPs, including any approved calibration revisions.
3. `annotation_history_<annotator>.csv` — the saved history of initial submissions and later edits.

Please submit the files with their original filenames. You do **not** need to submit `calibration_current_<annotator>.csv` or the protocol JSON unless the research team specifically asks for them.

Run the progress/output cell below before submitting and confirm that it reports **75/75 calibration pairs**, **1,725/1,725 main pairs**, and **1,800/1,800 overall**.


# 7. Progress and output files

This section reports completion counts and saved-file locations. It does not show choice distributions or relationships among your ratings, so that feedback cannot influence later judgments.


In [ ]:
# ============================================================
# PROGRESS / OUTPUT CHECK
# ============================================================
print(f"Annotator: {ANNOTATOR_ID}")
print(f"Calibration saved : {completed(CAL_TASKS)} / {len(CAL_TASKS)}")
print(f"Main saved        : {completed(MAIN_TASKS)} / {len(MAIN_TASKS)}")
print(f"Overall           : {len(RECORDS)} / {len(ALL_TASKS)}")
print()
print(f"Current annotations : {OUT_CSV}")
print(f"Full edit history   : {HISTORY_CSV}")
print(f"Calibration baseline: {CAL_BASELINE_CSV if CAL_BASELINE_CSV.exists() else 'not yet created'}")
print(f"Calibration current : {CAL_CURRENT_CSV if CAL_CURRENT_CSV.exists() else 'not yet created'}")
print(f"Protocol JSON       : {PROTOCOL_JSON}")

if completed(CAL_TASKS) == 75 and completed(MAIN_TASKS) == 1725:
    print("\nAll 1,800 pair assignments are complete.")
    print("\nSubmit these three files from annotations/:")
    print(f"  1. {CAL_BASELINE_CSV.name}")
    print(f"  2. {OUT_CSV.name}")
    print(f"  3. {HISTORY_CSV.name}")
